# Composite-Signal Percentage Strategy Backtest

Run the same composite stock analysis used by `03_index_search.ipynb` through historical data, then manage each resulting position with entry-relative exits. Set the ticker list, date range, analysis warm-up, sizing, costs, and risk limits in **Block 1**, then run all cells.

The notebook downloads and caches daily OHLCV data, builds point-in-time composite labels, and compares the strategy with buy-and-hold for the same ticker and period. Signals are acted on only at the following session's open.

> This is a daily-bar research backtest, not an execution simulator or financial advice. When both the target and stop are touched during one candle, the configured tie policy determines the simulated fill.


In [1]:
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from backtesting import (
    build_composite_signal_history,
    export_backtest_results,
    load_price_history_cached,
    plot_backtest_results,
    prepare_backtest_results,
    prepare_trade_details,
    run_price_intent_backtest,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.options.display.float_format = "{:,.2f}".format


## Block 1: Inputs

`start_date` begins the historical analysis window and `end_date` is inclusive; `end_date = None` means the latest available session. The first `trailing_days` completed trading sessions are analysis-only, and the earliest possible fill is the following session's open.

Only the requested ticker histories are loaded; no benchmark or index data is downloaded or calculated.


In [2]:
# Instruments and single backtest period
ticker = ["CVX"]  # Each ticker gets an independent backtest and its own initial_capital.
start_date = "2000-01-01"  # First date included in the analysis warm-up.
end_date = None              # Last date included; None means the latest available session.
trailing_days = 253            # Completed trading sessions required before the first signal can be acted on.

# Composite-signal strategy. BUY/SELL/HOLD labels use the shared analysis configuration.
include_sentiment = False          # Historical sentiment snapshots are unavailable, so keep this False.
allow_short_selling = False        # True permits WEAK/STRONG SELL labels to open short positions.
profit_target_pct = 0.10           # Long target is 10% above entry; short target is 10% below.
trailing_stop_pct = 0.99           # Trail by 1% from the best completed high/low since entry.

# Capital, position, and order limits
initial_capital = 100.00  # Starting cash assigned separately to every ticker.
position_size_pct = 1  # Fraction of current cash committed per entry: 1.00 = 100%, 0.25 = 25%.
max_shares = None         # Maximum shares per position; None leaves the position limited only by cash.
whole_shares = False      # False permits fractional shares; True rounds the quantity down.
max_entries = 1000        # Maximum number of successfully opened positions during the period.
allow_reentry = True      # True permits another entry after a position has exited.
cooldown_days = 5         # Full trading sessions to wait after an exit before another entry.
min_holding_days = 30      # Zero allows targets and stops to protect the position immediately.
max_holding_days = 1000   # Last priority: exit at close after N sessions only when no stop or target exits first.

# Execution assumptions
commission_per_order = 0.00  # Flat cash fee charged on every entry and exit order.
slippage_bps = 0.0           # Adverse fill adjustment: 10 basis points = 0.10% per order.
same_day_exit_priority = "target"  # If target and stop are both touched after entry, choose "stop" or "target".
entry_bar_exit_policy = "target"   # Daily bars cannot reveal whether an entry-candle target or stop happened first.
level_update_mode = "entry"      # Lock each trade's percentage-derived target at entry.
allow_stop_widening = False         # False prevents an active stop from moving farther away from the position.
force_exit_at_end = False           # Separate override: True closes on the final close even if minimum hold is unmet.

# Cache and optional exports
cache_dir = "cache"            # Directory for downloaded ticker price history.
cache_max_age_hours = 24        # Cached files older than this are considered stale.
refresh_price_cache = True      # True requests fresh prices even when a usable cache file exists.
export_results = False          # True writes result tables to CSV files under output/.
save_chart = False              # True saves each chart under output/ in addition to displaying it.


## Block 2: Load the single-period market data

Each ticker is loaded once for the requested period. The first `trailing_days` sessions seed the causal analysis; they cannot generate a fill. No benchmark, index membership, or training split is used.


In [3]:
normalized_tickers = list(dict.fromkeys(
    str(symbol).strip().upper()
    for symbol in ticker
    if symbol is not None and str(symbol).strip()
))
if not normalized_tickers:
    raise ValueError("ticker must contain at least one valid symbol")

start_ts = pd.Timestamp(start_date).normalize()
end_ts = (
    pd.Timestamp.today().normalize()
    if end_date is None
    else pd.Timestamp(end_date).normalize()
)
if end_ts < start_ts:
    raise ValueError("end_date must be on or after start_date")
if int(trailing_days) < 1:
    raise ValueError("trailing_days must be at least 1")

market_data_by_ticker = {}
cache_rows = []

for current_ticker in normalized_tickers:
    price_history, cache_meta = load_price_history_cached(
        symbol=current_ticker,
        start=start_date,
        end=end_date,
        cache_root=cache_dir,
        max_age_hours=cache_max_age_hours,
        force_refresh=refresh_price_cache,
    )
    if price_history.empty:
        raise ValueError(f"{current_ticker} has no rows in the selected period")
    if len(price_history) <= int(trailing_days):
        raise ValueError(
            f"{current_ticker} needs more than {trailing_days} trading sessions: "
            f"{trailing_days} completed sessions for analysis plus an execution session"
        )

    market_data_by_ticker[current_ticker] = {
        "price_history": price_history,
        "ticker_cache_meta": cache_meta,
    }
    cache_rows.append(dict(cache_meta))
    print(
        f"Loaded {len(price_history):,} {current_ticker} sessions from "
        f"{price_history['Date'].min():%Y-%m-%d} to "
        f"{price_history['Date'].max():%Y-%m-%d}; earliest possible trade "
        f"{price_history.iloc[int(trailing_days)]['Date']:%Y-%m-%d}."
    )

display(pd.DataFrame(cache_rows))


Loaded 6,694 CVX sessions from 2000-01-03 to 2026-08-14; earliest possible trade 2001-01-03.


,symbol,rows,cache_used,stale_fallback,cache_age_hours,cache_file
0,CVX,6694,False,False,0.00,cache\price_history\CVX_20000101_latest_1d.csv


## Block 3: Run the strategy

Each ticker runs independently with its own `initial_capital`. While flat, the algorithm reads the prior completed session's composite label and may enter at the next open. While invested, analysis labels are ignored and the position is managed by its target, trailing stop, and holding limit. After an exit, the exit session's completed analysis can drive a later entry, subject to `cooldown_days`.

Execution rules:

- `WEAK BUY` and `STRONG BUY` can open longs; sell labels can open shorts only when explicitly enabled.
- `HOLD` and `INSUFFICIENT DATA` remain in cash and are reassessed after each completed session.
- Entry fills at the session opening after the signal, adjusted for slippage.
- Long target = entry fill x (1 + `profit_target_pct`).
- The trailing stop follows the highest completed high by `trailing_stop_pct`.
- Short trades use the inverse calculations when explicitly enabled.
- First, `min_holding_days` must be reached; at zero, protective exits are active immediately.
- After the minimum hold, the trailing stop and profit target are evaluated first.
- A gap through a target or stop fills at the opening price before slippage.
- If target and stop are both touched in one later candle, `same_day_exit_priority` decides the result.
- `max_holding_days` is checked last and exits only if no stop or target applies on that session.
- Open positions are marked to each close and optionally liquidated on the final session.


In [ ]:
backtests_by_ticker = {}
actual_trade_level_frames = []
for current_ticker in normalized_tickers:
    market_data = market_data_by_ticker[current_ticker]
    composite_analysis = build_composite_signal_history(
        history=market_data["price_history"],
        ticker=current_ticker,
        trailing_days=trailing_days,
        include_sentiment=include_sentiment,
    )
    market_data["composite_analysis"] = composite_analysis
    backtest = run_price_intent_backtest(
        history=market_data["price_history"],
        benchmark=None,
        entry_limit=None,
        target=None,
        capital=initial_capital,
        size_pct=position_size_pct,
        share_limit=max_shares,
        use_whole_shares=whole_shares,
        entries_limit=max_entries,
        reentry=allow_reentry,
        wait_days=cooldown_days,
        minimum_holding_days=min_holding_days,
        holding_limit=max_holding_days,
        trailing_pct=trailing_stop_pct,
        fee=commission_per_order,
        slip_bps=slippage_bps,
        priority=same_day_exit_priority,
        entry_bar_exit_policy=entry_bar_exit_policy,
        level_update_mode=level_update_mode,
        allow_stop_widening=allow_stop_widening,
        liquidate_at_end=force_exit_at_end,
        allow_short=allow_short_selling,
        entry_at_market=True,
        entry_signal_text=composite_analysis["Signal_Text"],
        target_pct=profit_target_pct,
    )
    backtests_by_ticker[current_ticker] = backtest

    trade_levels = prepare_trade_details(backtest, ticker=current_ticker)
    if not trade_levels.empty:
        trade_levels = trade_levels.copy()
        trade_levels["trailing_stop_pct_points"] = (
            trade_levels["trailing_stop_pct"] * 100
        )
        actual_trade_level_frames.append(trade_levels[[
            "ticker", "trade_number", "side", "status",
            "entry_date", "entry_price", "planned_target_price",
            "trailing_stop_pct_points", "exit_date", "exit_price",
            "exit_reason", "return_pct",
        ]])
    print(f"Completed backtest: {current_ticker}")

actual_trade_levels = (
    pd.concat(actual_trade_level_frames, ignore_index=True)
    if actual_trade_level_frames
    else pd.DataFrame({"message": ["No entries were executed."]})
)
strategy_settings = pd.Series({
    "entry_rule": (
        "Enter long/short at the next open from the prior-session composite label"
        if allow_short_selling
        else "Enter long at the next open from prior-session composite BUY labels"
    ),
    "analysis_warmup_sessions": trailing_days,
    "signal_labels": "STRONG/WEAK BUY, HOLD, STRONG/WEAK SELL",
    "position_signal_policy": "ignore analysis labels until the position exits",
    "exit_priority": "minimum hold -> stops/target -> maximum hold",
    "profit_target_pct_from_entry": profit_target_pct * 100,
    "trailing_stop_pct": trailing_stop_pct * 100,
    "initial_capital_per_ticker": initial_capital,
    "position_size_pct": position_size_pct * 100,
    "cooldown_sessions": cooldown_days,
    "min_holding_sessions": min_holding_days,
    "max_holding_sessions": max_holding_days,
    "commission_per_order": commission_per_order,
    "slippage_bps": slippage_bps,
}, name="value").to_frame()

display(strategy_settings)
display(actual_trade_levels)


## Block 4: Results

The combined summary contains one row per ticker for the selected period. The order and round-trip tables include a ticker column so every fill remains attributable to its backtest.


In [ ]:
results_by_ticker = {}
summary_frames = []
transaction_frames = []
round_trip_frames = []
equity_frames = []
exported_files_by_ticker = {}
for current_ticker, backtest in backtests_by_ticker.items():
    ticker_results = prepare_backtest_results(backtest)
    results_by_ticker[current_ticker] = ticker_results
    for source_name, destination in [
        ("summary", summary_frames),
        ("transactions", transaction_frames),
        ("round_trips", round_trip_frames),
        ("equity_curve", equity_frames),
    ]:
        labeled_frame = ticker_results[source_name].copy()
        labeled_frame.insert(0, "ticker", current_ticker)
        destination.append(labeled_frame)

    if export_results:
        exported_files_by_ticker[current_ticker] = export_backtest_results(
            backtest, current_ticker, start_date
        )

summary = pd.concat(summary_frames, ignore_index=True)
transactions = pd.concat(transaction_frames, ignore_index=True)
round_trips = pd.concat(round_trip_frames, ignore_index=True)
equity_curve = pd.concat(equity_frames, ignore_index=True)

display(summary)
display(
    transactions
    if not transactions.empty
    else pd.DataFrame({"message": ["No orders filled for any ticker."]})
)
display(
    round_trips
    if not round_trips.empty
    else pd.DataFrame({"message": ["No completed round trips for any ticker."]})
)

if export_results:
    export_directory = next(
        iter(next(iter(exported_files_by_ticker.values())).values())
    ).parent.resolve()
    print(
        f"Exported {len(exported_files_by_ticker)} ticker backtests to "
        f"{export_directory}"
    )


## Block 5: Price and performance charts


In [ ]:
figures_by_ticker = {}
chart_files_by_ticker = {}
for current_ticker, backtest in backtests_by_ticker.items():
    equity_levels = backtest["equity_curve"]
    fig, chart_file = plot_backtest_results(
        backtest=backtest,
        ticker=current_ticker,
        buy_price=None,
        sell_price=equity_levels["target"],
        stop_loss=equity_levels["active_stop"],
        initial_capital=initial_capital,
        chart_output_dir="output" if save_chart else None,
    )
    figures_by_ticker[current_ticker] = fig
    chart_files_by_ticker[current_ticker] = chart_file
    if chart_file is not None:
        print(f"Saved {current_ticker} chart to {chart_file.resolve()}")
    plt.show()


## Block 6: Simulated trade execution details

One row per simulated trade across every ticker, including the filled entry and exit, quantity, fees, entry-time target and risk levels, holding period, and realized result. These are historical research fillsâ€”not live orders or a recommendation to trade.


In [ ]:
trade_detail_frames = [
    prepare_trade_details(backtest, ticker=current_ticker)
    for current_ticker, backtest in backtests_by_ticker.items()
]
trade_details = pd.concat(trade_detail_frames, ignore_index=True)
trade_details = trade_details.drop(columns=[
    "planned_stop_price",
    "planned_risk_per_share",
    "planned_reward_risk_ratio",
])

if trade_details.empty:
    display(pd.DataFrame({"message": ["No simulated trade entries were executed."]}))
else:
    display(trade_details.sort_values(["entry_date", "ticker"]))

if export_results:
    for current_ticker, exported_files in exported_files_by_ticker.items():
        print(
            f"{current_ticker} trade-detail CSV: "
            f"{exported_files['trade_details'].resolve()}"
        )